# Análise exploratória — anúncios de veículos usados (EUA)

Objetivo: entender a qualidade dos dados de `vehicles_us.csv` e quais fatores mais se relacionam com o preço dos anúncios, antes de construir o painel em Streamlit (`app.py`).

In [1]:
import pandas as pd
import plotly.express as px

car_data = pd.read_csv('../vehicles_us.csv')
car_data.head()

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
0,9400,2011.0,bmw x5,good,6.0,gas,145000.0,automatic,SUV,NaN,1.0,2018-06-23,19
1,25500,NaN,ford f-150,good,6.0,gas,88705.0,automatic,pickup,white,1.0,2018-10-19,50
2,5500,2013.0,hyundai sonata,like new,4.0,gas,110000.0,automatic,sedan,red,NaN,2019-02-07,79
3,1500,2003.0,ford f-150,fair,8.0,gas,NaN,automatic,pickup,NaN,NaN,2019-03-22,9
4,14900,2017.0,chrysler 200,excellent,4.0,gas,80903.0,automatic,sedan,black,NaN,2019-04-02,28


## 1. Visão geral dos dados

In [2]:
car_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  str    
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   42258 non-null  str    
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 7.7 MB


In [3]:
car_data.describe()

,price,model_year,cylinders,odometer,is_4wd,days_listed
count,51525.000000,47906.000000,46265.000000,43633.000000,25572.0,51525.00000
mean,12132.464920,2009.750470,6.125235,115553.461738,1.0,39.55476
std,10040.803015,6.282065,1.660360,65094.611341,0.0,28.20427
min,1.000000,1908.000000,3.000000,0.000000,1.0,0.00000
25%,5000.000000,2006.000000,4.000000,70000.000000,1.0,19.00000
50%,9000.000000,2011.000000,6.000000,113000.000000,1.0,33.00000
75%,16839.000000,2014.000000,8.000000,155000.000000,1.0,53.00000
max,375000.000000,2019.000000,12.000000,990000.000000,1.0,271.00000


In [4]:
# Proporção de valores ausentes por coluna
car_data.isna().mean().sort_values(ascending=False).round(3)

is_4wd          0.504
paint_color     0.180
odometer        0.153
cylinders       0.102
model_year      0.070
condition       0.000
model           0.000
price           0.000
fuel            0.000
type            0.000
transmission    0.000
date_posted     0.000
days_listed     0.000
dtype: float64

In [5]:
car_data.duplicated().sum()

np.int64(0)

**Observações:**
- `is_4wd` tem ~50% de ausentes: a coluna só registra `1`, então ausente significa *não é 4x4*.
- `paint_color` (~18%), `odometer` (~15%), `cylinders` (~10%) e `model_year` (~7%) têm lacunas que podem ser preenchidas a partir de grupos semelhantes.
- Não há linhas duplicadas.

## 2. Tratamento de dados

In [6]:
df = car_data.copy()

df['date_posted'] = pd.to_datetime(df['date_posted'])
df['manufacturer'] = df['model'].str.split().str[0]

df['is_4wd'] = df['is_4wd'].fillna(0).astype(bool)
df['paint_color'] = df['paint_color'].fillna('unknown')

# Ano e cilindros: mediana do mesmo modelo
for col in ['model_year', 'cylinders']:
    df[col] = df[col].fillna(df.groupby('model')[col].transform('median'))

# Quilometragem: mediana do mesmo ano de fabricação
df['odometer'] = df['odometer'].fillna(df.groupby('model_year')['odometer'].transform('median'))

df['model_year'] = df['model_year'].round().astype('Int64')
df['cylinders'] = df['cylinders'].round().astype('Int64')

df.isna().sum()

price           0
model_year      0
model           0
condition       0
cylinders       0
fuel            0
odometer        1
transmission    0
type            0
paint_color     0
is_4wd          0
date_posted     0
days_listed     0
manufacturer    0
dtype: int64

## 3. Distribuição de preços

In [7]:
fig = px.histogram(df, x='price', nbins=100, title='Distribuição de preços (dados brutos)')
fig.show()

In [8]:
print('Anúncios com preço abaixo de US$ 500:', (df['price'] < 500).sum())
df['price'].quantile([0.01, 0.05, 0.5, 0.95, 0.99])

Anúncios com preço abaixo de US$ 500: 1069


0.01        1.0
0.05     1950.0
0.50     9000.0
0.95    30300.0
0.99    43995.0
Name: price, dtype: float64

A distribuição é fortemente assimétrica à direita, com valores extremos (até US$ 375 mil) e mais de mil anúncios abaixo de US$ 500 (vários com preço US$ 1, provavelmente placeholders). Para as análises seguintes, removemos preços abaixo de US$ 500 e acima do percentil 99, além da quilometragem acima do percentil 99.

In [9]:
price_cap = df['price'].quantile(0.99)
odometer_cap = df['odometer'].quantile(0.99)
clean = df[df['price'].between(500, price_cap) & (df['odometer'] <= odometer_cap)]

print(f'{len(df)} -> {len(clean)} anúncios')
fig = px.histogram(clean, x='price', nbins=60, title='Distribuição de preços (sem outliers)')
fig.show()

51525 -> 49432 anúncios


## 4. Quilometragem e idade

In [10]:
fig = px.histogram(clean, x='odometer', nbins=60, title='Distribuição da quilometragem')
fig.show()

In [11]:
fig = px.histogram(clean, x='model_year', nbins=60, title='Ano do modelo dos veículos anunciados')
fig.show()

## 5. Relação entre preço e outras variáveis

In [12]:
fig = px.scatter(clean, x='odometer', y='price', color='condition', opacity=0.4,
                 title='Preço × quilometragem por condição')
fig.show()

In [13]:
fig = px.scatter(clean, x='model_year', y='price', opacity=0.3, title='Preço × ano do modelo')
fig.show()

In [14]:
clean[['price', 'odometer', 'model_year', 'days_listed']].corr().round(2)

,price,odometer,model_year,days_listed
price,1.00,-0.51,0.49,-0.0
odometer,-0.51,1.00,-0.49,0.0
model_year,0.49,-0.49,1.00,-0.0
days_listed,-0.00,0.00,-0.00,1.0


## 6. Preço por categoria

In [15]:
order = ['new', 'like new', 'excellent', 'good', 'fair', 'salvage']
fig = px.box(clean, x='condition', y='price', category_orders={'condition': order},
             title='Preço por condição do veículo')
fig.show()

In [16]:
median_by_type = clean.groupby('type')['price'].median().sort_values(ascending=False).reset_index()
fig = px.bar(median_by_type, x='type', y='price', title='Preço mediano por tipo de veículo')
fig.show()

In [17]:
fig = px.histogram(clean, x='manufacturer', title='Anúncios por fabricante').update_xaxes(categoryorder='total descending')
fig.show()

In [18]:
fig = px.histogram(clean, x='days_listed', nbins=60, title='Dias até o anúncio sair do ar')
fig.show()

## 7. Conclusões

- **Qualidade dos dados:** as lacunas foram preenchidas com medianas de grupos semelhantes (modelo ou ano), preservando todas as linhas; ~4% dos anúncios foram tratados como outliers de preço/quilometragem.
- **Preço:** mediana em torno de US$ 9 mil, com distribuição assimétrica; a maioria dos anúncios está entre ~US$ 2 mil e ~US$ 30 mil.
- **Idade e uso:** preço tem correlação moderada positiva com o ano do modelo (~0,5) e negativa com a quilometragem (~-0,5) — carros mais novos e menos rodados valem mais.
- **Condição:** o preço mediano cresce de forma consistente de *salvage/fair* (~US$ 2,5 mil) até *new* (~US$ 22 mil).
- **Tipo:** picapes e caminhões (*pickup*, *truck*) têm os maiores preços medianos; *hatchbacks* e sedãs, os menores.
- **Fabricantes:** Ford e Chevrolet concentram a maior parte dos anúncios.
- **Tempo de anúncio:** `days_listed` praticamente não se relaciona com o preço (correlação ≈ 0); a mediana é de cerca de 33 dias.

Esses achados orientam o painel: histogramas para as distribuições, dispersão para preço × quilometragem/ano e comparação de preço mediano por categoria.